# Interpretable Fairness-Aware ML Toolkit Demo

This notebook demonstrates the complete workflow of the Interpretable Fairness-Aware ML Toolkit:
1. Train a classification model
2. Generate explanations using SHAP
3. Create counterfactual explanations
4. Evaluate fairness metrics
5. Generate comprehensive audit report

In [ ]:
# Install required packages
# !pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Import toolkit modules
import explainability
import counterfactuals
import fairness
import report

print("Toolkit modules imported successfully!")

## 1. Create and Prepare Dataset

In [ ]:
# Generate synthetic dataset with bias
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=8,
    n_redundant=2,
    n_clusters_per_class=2,
    random_state=42
)

# Create sensitive attribute (e.g., gender, race)
# Introduce bias: group 1 has higher positive rate
np.random.seed(42)
sensitive_attr = np.random.choice([0, 1], size=len(X), p=[0.6, 0.4])

# Introduce bias in the dataset
bias_factor = 0.3
for i in range(len(X)):
    if sensitive_attr[i] == 1:  # Privileged group
        # Increase probability of positive outcome
        if y[i] == 0 and np.random.random() < bias_factor:
            y[i] = 1

print(f"Dataset shape: {X.shape}")
print(f"Positive rate overall: {np.mean(y):.3f}")
print(f"Positive rate group 0: {np.mean(y[sensitive_attr == 0]):.3f}")
print(f"Positive rate group 1: {np.mean(y[sensitive_attr == 1]):.3f}")

In [ ]:
# Create feature names
feature_names = [f'feature_{i}' for i in range(X.shape[1])]
feature_names += ['sensitive_attr']

# Combine features with sensitive attribute
X_with_sensitive = np.column_stack([X, sensitive_attr])

# Split data
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X_with_sensitive, y, sensitive_attr, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

## 2. Train Classification Model

In [ ]:
# Train Random Forest classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluate model
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {accuracy:.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Check bias in predictions
print(f"\nPrediction bias analysis:")
print(f"Positive prediction rate overall: {np.mean(y_pred):.3f}")
print(f"Positive prediction rate group 0: {np.mean(y_pred[sens_test == 0]):.3f}")
print(f"Positive prediction rate group 1: {np.mean(y_pred[sens_test == 1]):.3f}")

## 3. Generate Explanations with SHAP

In [ ]:
# Initialize explainer
explainer = explainability.Explainer(
    model=model, 
    model_type="sklearn",
    background_data=X_train[:100]  # Use subset for background
)

# Generate explanations for a few test instances
test_instances = X_test[:5]
explanations = explainer.explain_instance(test_instances, method="shap")

print("Generated SHAP explanations for test instances")
print(f"Explanation keys: {list(explanations.keys())}")

In [ ]:
# Visualize feature importance
if 'shap_values' in explanations:
    plt.figure(figsize=(10, 6))
    explainer.plot_summary(explanations, feature_names=feature_names)
    plt.title("SHAP Feature Importance Summary")
    plt.show()
else:
    print("No SHAP values available for plotting")

## 4. Generate Counterfactual Explanations

In [ ]:
# Initialize counterfactual generator
cf_generator = counterfactuals.CounterfactualGenerator(
    model=model,
    X_background=X_train,
    y_background=y_train,
    feature_names=feature_names,
    protected_features=['sensitive_attr']  # Don't modify sensitive attribute
)

# Generate counterfactual for a test instance
test_idx = 0
test_instance = X_test[test_idx]
test_prediction = y_pred[test_idx]

cf_result = cf_generator.generate_counterfactual(
    X=test_instance,
    y_original=test_prediction,
    max_distance=1.0,
    num_candidates=3
)

print(f"Original prediction: {test_prediction}")
print(f"Number of counterfactuals found: {cf_result['num_found']}")

if cf_result['counterfactuals']:
    cf = cf_result['counterfactuals'][0]
    print(f"Counterfactual prediction: {cf['prediction']}")
    print(f"Distance from original: {cf['distance']:.3f}")
    
    # Show feature changes
    print("\nFeature changes:")
    for feat_name, change in cf['changes'].items():
        if abs(change['difference']) > 0.01:
            print(f"{feat_name}: {change['original']:.3f} → {change['counterfactual']:.3f} (Δ{change['difference']:.3f})")

## 5. Evaluate Fairness Metrics

In [ ]:
# Initialize fairness auditor
auditor = fairness.FairnessAuditor(
    sensitive_attrs={'sensitive_attr': sens_test},
    privileged_groups={'sensitive_attr': 1},  # Group 1 is privileged
    attribute_names={'sensitive_attr': 'Sensitive Attribute'}
)

# Perform comprehensive fairness audit
fairness_results = auditor.audit(
    y_true=y_test,
    y_pred=y_pred,
    y_prob=y_prob
)

print("Fairness audit completed!")
print(f"Overall fairness score: {fairness_results['fairness_summary']['overall_fairness_score']:.3f}")

In [ ]:
# Display key fairness metrics
fairness_metrics = fairness_results['fairness_metrics']['sensitive_attr']

print("Key Fairness Metrics:")
print(f"Statistical Parity Difference: {fairness_metrics['statistical_parity_difference']:.3f}")
print(f"Equal Opportunity Difference: {fairness_metrics['equal_opportunity_difference']:.3f}")
print(f"Disparate Impact Ratio: {fairness_metrics['disparate_impact_ratio']:.3f}")
print(f"Accuracy Difference: {fairness_metrics['accuracy_difference']:.3f}")

# Display equalized odds
print("\nEqualized Odds:")
odds = fairness_metrics['equalized_odds']
for group, metrics in odds.items():
    print(f"Group {group}: TPR={metrics['tpr']:.3f}, FPR={metrics['fpr']:.3f}")

In [ ]:
# Display fairness assessment summary
summary = fairness_results['fairness_summary']

print("Fairness Assessment:")
if summary['critical_issues']:
    print("\n🚨 Critical Issues:")
    for issue in summary['critical_issues']:
        print(f"  - {issue}")

if summary['warnings']:
    print("\n⚠️  Warnings:")
    for warning in summary['warnings']:
        print(f"  - {warning}")

if summary['passed_checks']:
    print("\n✅ Passed Checks:")
    for check in summary['passed_checks']:
        print(f"  - {check}")

## 6. Generate Comprehensive Audit Report

In [ ]:
# Prepare model information
model_info = {
    'name': 'RandomForest Classifier',
    'type': 'sklearn.ensemble.RandomForestClassifier',
    'features': len(feature_names),
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

# Generate comprehensive audit report
report_path = report.generate_audit_report(
    fairness_results=fairness_results,
    explainability_results=explanations,
    counterfactual_results=cf_result,
    model_info=model_info,
    output_path="../audit_report.html"
)

print(f"Audit report generated: {report_path}")
print("Open the HTML file in your browser to view the comprehensive report.")

## 7. Generate Recommendations

In [ ]:
# Get recommendations from the auditor
recommendations = auditor.generate_recommendations()

print("📋 Recommendations:")
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

## 8. Summary

This demo demonstrated the complete workflow of the Interpretable Fairness-Aware ML Toolkit:

1. **Data Preparation**: Created a biased dataset with sensitive attributes
2. **Model Training**: Trained a Random Forest classifier
3. **Explainability**: Generated SHAP explanations to understand feature importance
4. **Counterfactuals**: Created counterfactual explanations while protecting sensitive attributes
5. **Fairness Analysis**: Evaluated multiple fairness metrics and identified bias
6. **Audit Report**: Generated a comprehensive HTML report with visualizations
7. **Recommendations**: Received actionable suggestions for improving fairness

The toolkit provides:
- ✅ **Explainability**: SHAP and Captum integration for model interpretability
- ✅ **Counterfactuals**: Nearest neighbor counterfactual generation with protected features
- ✅ **Fairness Metrics**: Comprehensive group fairness evaluation
- ✅ **Audit Reports**: Professional HTML reports with visualizations
- ✅ **Policy Ready**: Audit-ready documentation for compliance teams

This makes it valuable for organizations needing to ensure their ML models are both interpretable and fair, meeting regulatory and ethical requirements.